# Exploracion - QuickBooks

Analisis exploratorio para preparar modelado de pronostico: calidad, semantica de producto, vigencia y estabilidad temporal.


In [ ]:
from pathlib import Path
import re
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

CWD = Path.cwd().resolve()
candidates = [CWD, CWD.parent, CWD / "03_modelado" / "proyecto_ml_experimentos"]
ROOT = next((p for p in candidates if (p / "src" / "datasets_postgres.py").exists()), None)
if ROOT is None:
    raise RuntimeError("No se encontro la raiz de proyecto_ml_experimentos.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

for m in ["src", "src.datasets_postgres", "src.postgres_loader"]:
    if m in sys.modules:
        del sys.modules[m]

from src.datasets_postgres import load_forecasting_dataset
from src.postgres_loader import load_dwh_query, load_quickbooks_query

charts_dir = ROOT.parents[1] / "05_evidencias" / "graficas"
charts_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
df_monthly = load_forecasting_dataset()

df_raw = load_quickbooks_query(
    """
    SELECT producto, fecha, qty_planificada, qty_fabricada
    FROM quickbooks.produccion
    WHERE producto IS NOT NULL AND fecha IS NOT NULL
    """
)

df_raw["producto"] = df_raw["producto"].astype(str).str.strip()
df_raw["fecha"] = pd.to_datetime(df_raw["fecha"], errors="coerce")
for c in ["qty_planificada", "qty_fabricada"]:
    df_raw[c] = pd.to_numeric(df_raw[c], errors="coerce").fillna(0)

df_raw = df_raw[(df_raw["producto"] != "") & df_raw["fecha"].notna()].copy()

display(df_monthly.head())
display(df_raw.head())


In [ ]:
def tipo_producto(name):
    s = str(name).upper().strip()
    if s.startswith("PP"):
        return "PP"
    if s.startswith("PT"):
        return "PT"
    return "OTRO"


def producto_norm(name):
    s = str(name).upper()
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"^\*+", "", s).strip()
    s = re.sub(r"\bEXTR\b", "EXT", s)
    s = re.sub(r"[^A-Z0-9 ]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


df_raw["tipo_producto"] = df_raw["producto"].apply(tipo_producto)
df_raw["producto_base_norm"] = df_raw["producto"].apply(producto_norm)

raw_rows = len(df_raw)
raw_prod = df_raw["producto"].nunique()
raw_prod_norm = df_raw["producto_base_norm"].nunique()
dup_exact = int(df_raw.duplicated(subset=["producto", "fecha", "qty_planificada", "qty_fabricada"]).sum())
gap_sem = int(raw_prod - raw_prod_norm)

display(
    Markdown(
        f"""
## Calidad base QuickBooks
- Filas crudas: **{raw_rows:,}**
- Productos crudos: **{raw_prod:,}**
- Productos normalizados: **{raw_prod_norm:,}**
- Brecha semantica (crudo-normalizado): **{gap_sem:,}**
- Duplicados exactos (`producto,fecha,qty_planificada,qty_fabricada`): **{dup_exact:,}**

## Dataset mensual de modelado
- Filas: **{len(df_monthly):,}**
- Productos: **{df_monthly['producto'].nunique():,}**
- Periodos: **{df_monthly['periodo'].nunique():,}**
- Rango: **{df_monthly['periodo'].min().date()}** a **{df_monthly['periodo'].max().date()}**
"""
    )
)

display(df_raw["tipo_producto"].value_counts(dropna=False).rename_axis("tipo").to_frame("conteo"))


In [ ]:
max_period = df_monthly["periodo"].max()
last_prod = df_monthly.groupby("producto", as_index=False)["periodo"].max().rename(columns={"periodo": "ultimo_periodo"})
last_prod["meses_desfase"] = (
    max_period.to_period("M") - last_prod["ultimo_periodo"].dt.to_period("M")
).apply(lambda x: x.n)
last_prod["es_vigente_3m"] = last_prod["meses_desfase"] <= 2

display(
    Markdown(
        f"""
## Vigencia operativa
- Periodo global mas reciente: **{max_period.date()}**
- Productos vigentes (<=2 meses): **{int(last_prod['es_vigente_3m'].sum()):,}**
- Productos fuera de ventana: **{int((~last_prod['es_vigente_3m']).sum()):,}**
"""
    )
)

display(last_prod.sort_values("meses_desfase", ascending=False).head(15))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_monthly["qty_fabricada"], bins=40, ax=axes[0], color="#1f77b4")
axes[0].set_title("Distribucion qty_fabricada")
sns.histplot(df_monthly["qty_planificada"], bins=40, ax=axes[1], color="#ff7f0e")
axes[1].set_title("Distribucion qty_planificada")
plt.tight_layout()
plt.savefig(charts_dir / "eda_quickbooks_distribuciones_qty.png", dpi=150)
plt.show()

monthly = (
    df_monthly.groupby("periodo", as_index=False)[["qty_fabricada", "qty_planificada"]]
    .sum()
    .sort_values("periodo")
)
plt.figure(figsize=(12, 4))
plt.plot(monthly["periodo"], monthly["qty_fabricada"], marker="o", label="Fabricada")
plt.plot(monthly["periodo"], monthly["qty_planificada"], marker="o", label="Planificada")
plt.title("Tendencia mensual total QuickBooks")
plt.legend()
plt.tight_layout()
plt.savefig(charts_dir / "eda_quickbooks_tendencia_mensual.png", dpi=150)
plt.show()


In [ ]:
stats_prod = (
    df_monthly.groupby("producto", as_index=False)
    .agg(
        media_fabricada=("qty_fabricada", "mean"),
        std_fabricada=("qty_fabricada", "std"),
        media_planificada=("qty_planificada", "mean"),
        periodos=("periodo", "nunique"),
    )
)
stats_prod["std_fabricada"] = stats_prod["std_fabricada"].fillna(0)
stats_prod["cv_fabricada"] = np.where(
    stats_prod["media_fabricada"] > 0,
    stats_prod["std_fabricada"] / stats_prod["media_fabricada"],
    np.nan,
)

display(Markdown("## Top productos por volumen"))
display(stats_prod.sort_values("media_fabricada", ascending=False).head(15))

display(Markdown("## Top productos por volatilidad (CV)"))
display(stats_prod.dropna(subset=["cv_fabricada"]).sort_values("cv_fabricada", ascending=False).head(15))


In [ ]:
gold_prod = load_dwh_query(
    """
    SELECT cliente, qty_total_planificada, qty_total_despachada, tasa_cumplimiento, num_ordenes
    FROM gold.kpis_produccion
    """
)
for c in ["qty_total_planificada", "qty_total_despachada", "tasa_cumplimiento", "num_ordenes"]:
    gold_prod[c] = pd.to_numeric(gold_prod[c], errors="coerce").fillna(0)

display(
    Markdown(
        f"""
## Contraste con Gold
- Filas `gold.kpis_produccion`: **{len(gold_prod):,}**
- Cumplimiento promedio: **{gold_prod['tasa_cumplimiento'].mean():.2f}**
- Cumplimiento max: **{gold_prod['tasa_cumplimiento'].max():.2f}**
"""
    )
)

display(gold_prod.describe(include="all").T)


In [ ]:
no_vigentes = int((~last_prod["es_vigente_3m"]).sum())
riesgo_sem = "ALTO" if gap_sem > 0 else "BAJO"
riesgo_vig = "ALTO" if no_vigentes > 0 else "BAJO"

display(
    Markdown(
        f"""
## Conclusiones para modelado QuickBooks
1. Usar `load_forecasting_dataset` como base principal para entrenamiento de pronostico.
2. Riesgo semantico de productos: **{riesgo_sem}**.
3. Riesgo de vigencia operativa: **{riesgo_vig}** (fuera de ventana: **{no_vigentes:,}**).
4. Regla sugerida: excluir o separar SKU fuera de ventana (>2 meses) en plan operativo.
5. Gold sirve para contraste ejecutivo, no para reemplazar la granularidad de entrenamiento.

Graficas guardadas en: `{charts_dir}`
"""
    )
)
